# Byte-level BPE（GPT-2 / GPT-3 算法）

源码导航：[`core/tokenizer/byte_bpe.py`](../../../core/tokenizer/byte_bpe.py)。

## 1. 理论背景

**Byte-level BPE（BBPE）** 是字符级 BPE 的扩展：将任意输入文本首先编码为 UTF-8 字节流（取值范围 0–255），在字节级别执行 BPE 合并，从而将初始词表固定为 256 个字节单元。

BBPE 的核心性质：

1. **无 OOV**：任何有效 Unicode 字符串均可无损编码为 UTF-8 字节序列，256 个字节单元构成完备的初始词表，不存在未登录词（OOV）。
2. **跨语言通用**：中文、日文、Emoji 等扩展字符无需特殊处理，均通过多字节序列表示。
3. **词表效率**：在字节级起步的合并覆盖面更广，最终词表对各类语言的编码效率均匀。

### 字节到可见字符的映射

直接操作原始字节（含控制字符 `\x00`–`\x1f`、`\x7f` 等）调试困难。OpenAI 引入一个**字节 ↔ 可打印 Unicode 字符**的双向映射表 `bytes_to_unicode()`，将 256 个字节一一映射到 256 个可打印字符：

- 字节 `33–126`（`!` 到 `~`）和 `161–255` 直接保留为对应 Unicode 字符
- 其余字节（`0–32`、`127–160`）映射到 `256` 起步的 Unicode 码点

这使 BPE 合并步骤完全在可打印字符域中进行，训练完成后仍可通过逆映射还原原始字节。

In [ ]:
def bytes_to_unicode():
    """将 256 个字节映射到可打印的 Unicode 字符"""
    bs = (
        list(range(ord("!"), ord("~") + 1))
        + list(range(ord("¡"), ord("¬") + 1))
        + list(range(ord("®"), ord("ÿ") + 1))
    )
    cs = list(bs)
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

b2u = bytes_to_unicode()
u2b = {v: k for k, v in b2u.items()}

# 测试一下映射
sample_byte = ord(' ') # 空格字节 32
mapped_char = b2u[sample_byte]
print(f"字节 {sample_byte} 被映射到了字符: '{mapped_char}' (Unicode: {ord(mapped_char)})")

## 3. 训练流程拆解

### 3.1 编码为映射后的字符序列
 BBPE 训练的第一步是将文本转为 UTF-8 字节，再查表转为映射字符。

源码对应：[`ByteBPETokenizer.train`](../../../core/tokenizer/byte_bpe.py#L110)


In [ ]:
text = "hello 世界"
# 1. UTF-8 编码
raw_bytes = text.encode("utf-8")
print(f"原始字节流: {list(raw_bytes)}")

# 2. 映射为可见字符
mapped_chars = [b2u[b] for b in raw_bytes]
print(f"映射后的字符序列: {''.join(mapped_chars)}")

## 4. 编解码全流程演示

我们使用源码中的逻辑来模拟一次完整的 BBPE 往返转换。

源码对应：[`encode`](../../../core/tokenizer/byte_bpe.py#L160) 和 [`decode`](../../../core/tokenizer/byte_bpe.py#L173)


In [ ]:
def demo_bbpe_encode(text, b2u):
    # 模拟编码流程：Text -> Bytes -> Mapped Chars -> (BPE Merges)
    raw_bytes = text.encode("utf-8")
    uchars = [b2u[b] for b in raw_bytes]
    # 在实际实现中，这里会调用 self._bpe(uchars) 进行合并
    return uchars

def demo_bbpe_decode(uchars, u2b):
    # 模拟解码流程：Mapped Chars -> Bytes -> Text
    byts = bytes([u2b[c] for c in uchars])
    return byts.decode("utf-8")

original_text = "GPT-2 🤖"
encoded = demo_bbpe_encode(original_text, b2u)
decoded = demo_bbpe_decode(encoded, u2b)

print(f"原文: {original_text}")
print(f"中间表示 (uchars): {encoded}")
print(f"还原: {decoded}")
print(f"无损还原成功? {original_text == decoded}")

## 5. 工程实现提示

在源码 [`core/tokenizer/byte_bpe.py`](../../../core/tokenizer/byte_bpe.py) 中，你会看到：
- **`@lru_cache`**：用于加速 `bytes_to_unicode` 的计算。
- **特殊 Token 过滤**：在 `decode` 时会过滤掉 `<|endoftext|>`，因为它不在 256 个映射字节内。
- **预切分 (Pre-tokenization)**：BBPE 同样需要 GPT-2 风格的正则预切分，以防止跨词（如空格与单词）的不合理合并。

---

## 6. 与 OpenAI/HuggingFace 的关系

- 源码实现的算法逻辑与官方完全一致。
- **自训练词表**：即使算法相同，但在不同语料上训练出的 `merges` 和 `vocab` 会不同。
- **权重对齐**：如果你想使用 GPT-2 的官方 50257 权重，请使用 [`core/tokenizer/gpt2_bpe.py`](../../../core/tokenizer/gpt2_bpe.py)，它支持直接加载 `vocab.json`。

## 7. 延伸阅读与参考资料

### 核心论文 (Paper)
- **GPT-2 技术报告**: Radford et al., 2019. *Language Models are Unsupervised Multitask Learners* (主要介绍了字节级改进). [Link](https://openai.com/blog/better-language-models/)

### 优质博客 (Blog)
- **Andrej Karpathy**: *minBPE*. [GitHub](https://github.com/karpathy/minbpe) (一个极其精简的 BBPE 教学实现)
- **Hugging Face**: *Byte-Pair Encoding tokenization*. [Reference](https://huggingface.co/learn/nlp-course/chapter6/5)

### 代码库参考 (Code)
- **OpenAI 原始实现**: [openai/gpt-2/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)

---
> 父文档：[← 分词器总览](index.ipynb)